In [ ]:
# ==========================================
# 0. MOUNT GOOGLE DRIVE
# ==========================================
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive berhasil disambungkan.")
except ImportError:
    print("Tidak berjalan di Colab. Pastikan path lokal sesuai.")

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import torch
import torch.nn as nn
from torchvision.models import resnet50
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import torch.nn.functional as F
import pandas as pd

# ==========================================
# 2. KONFIGURASI FOLDER & DEVICE
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# Root folder berdasarkan struktur di gambar
ROOT_DIR = '/content/drive/MyDrive/Alzheimer_MRI_Models'

# ==========================================
# 3. DEFINISI CLASS DATASET
# ==========================================
class AlzheimerDataset(Dataset):
    def __init__(self, hf_data, transform=None):
        self.data = hf_data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item["image"].convert("RGB")
        label = item["label"]

        if self.transform:
            image = self.transform(image)

        return image, label, idx # Sesuai dengan format return di training script

# ==========================================
# 4. PERSIAPAN DATA UJI (TEST DATASET)
# ==========================================
print("\nMenyiapkan Data Test...")
hf_dataset = load_dataset("Falah/Alzheimer_MRI")
test_data = hf_dataset["test"]

# ResNet50 menggunakan resolusi standar 224x224
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_ds = AlzheimerDataset(test_data, base_transform)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

# ==========================================
# 5. INISIALISASI ARSITEKTUR MODEL (RESNET50)
# ==========================================
print("\nMembangun arsitektur ResNet50...")
model = resnet50(weights=None)

num_classes = 4
# Harus sama persis dengan modifikasi layer fc saat training
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, num_classes)
)
model = model.to(device)
model.eval()

# ==========================================
# 6. FUNGSI EVALUASI MASSAL
# ==========================================
def evaluate_models_in_folder(folder_path, method_name):
    results = []

    if not os.path.exists(folder_path):
        return results

    model_files = [f for f in os.listdir(folder_path) if f.endswith('.pth')]
    model_files.sort()

    for filename in model_files:
        filepath = os.path.join(folder_path, filename)

        # Ekstrak Skenario dan Epoch dari nama file (Format: S-01_best_epoch_12.pth)
        parts = filename.replace('.pth', '').split('_')
        scenario = parts[0] if len(parts) > 0 else filename
        best_epoch = parts[-1] if "epoch" in filename else "-"

        # 1. LOAD WEIGHTS
        model.load_state_dict(torch.load(filepath, map_location=device))

        # 2. PROSES INFERENCE DENGAN PROBABILITAS
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.to(device)
                outputs = model(images)

                probs = F.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        # 3. HITUNG METRIK
        acc = accuracy_score(all_labels, all_preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average='macro', zero_division=0
        )
        roc_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')

        # 4. SIMPAN HASIL
        results.append({
            "Folder / Method": method_name,
            "Scenario": scenario,
            "Best Epoch": best_epoch,
            "Accuracy": acc,
            "F1-Macro": f1,
            "Precision": precision,
            "Recall": recall,
            "ROC-AUC": roc_auc
        })

    return results

# ==========================================
# 7. JALANKAN EKSEKUSI DINAMIS & TAMPILKAN LAPORAN
# ==========================================
all_results = []

print(f"\nMencari subfolder di dalam: {ROOT_DIR}")
if os.path.exists(ROOT_DIR):
    # Ambil semua subfolder yang ada di dalam root directory
    subfolders = [f.path for f in os.scandir(ROOT_DIR) if f.is_dir()]
    subfolders.sort()

    for folder_path in subfolders:
        method_name = os.path.basename(folder_path)
        print(f"--- Memulai Evaluasi Folder: {method_name} ---")
        folder_results = evaluate_models_in_folder(folder_path, method_name)
        all_results.extend(folder_results)
else:
    print(f"Root folder tidak ditemukan: {ROOT_DIR}")

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)

    # Urutkan berdasarkan F1-Macro & ROC-AUC Tertinggi
    df_results_sorted = df_results.sort_values(by=['F1-Macro', 'ROC-AUC'], ascending=[False, False])

    print("\n\n================ KESIMPULAN HASIL EVALUASI RESNET50 ================")

    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_rows', None)

    display(df_results_sorted)
else:
    print("\nTidak ada model yang berhasil dievaluasi. Periksa kembali struktur file di Drive kamu.")

Mounted at /content/drive
Google Drive berhasil disambungkan.
Menggunakan device: cuda

Menyiapkan Data Test...


README.md:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

data/train-00000-of-00001-c08a401c53fe53(…): reconstructing file:   0%|          |  0.00B / 22.6MB            

data/train-00000-of-00001-c08a401c53fe53(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-44110b9df98c558(…): reconstructing file:   0%|          |  0.00B / 5.65MB            

data/test-00000-of-00001-44110b9df98c558(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1280 [00:00<?, ? examples/s]


Membangun arsitektur ResNet50...

Mencari subfolder di dalam: /content/drive/MyDrive/Alzheimer_MRI_Models
--- Memulai Evaluasi Folder: Freeze_All ---
--- Memulai Evaluasi Folder: Freeze_All_Sampler ---
--- Memulai Evaluasi Folder: Full_Tuning ---
--- Memulai Evaluasi Folder: Full_Tuning_Sampler ---
--- Memulai Evaluasi Folder: Unfreeze_Last_Layer ---
--- Memulai Evaluasi Folder: Unfreeze_Last_Layer_Sampler ---


================ KESIMPULAN HASIL EVALUASI RESNET50 ================


,Folder / Method,Scenario,Best Epoch,Accuracy,F1-Macro,Precision,Recall,ROC-AUC
13,Full_Tuning,S-14,24,0.9875,0.9910,0.9927,0.9893,0.9993
15,Full_Tuning,S-16,24,0.9875,0.9910,0.9927,0.9893,0.9993
12,Full_Tuning,S-13,25,0.9891,0.9898,0.9938,0.9862,0.9997
14,Full_Tuning,S-15,25,0.9891,0.9898,0.9938,0.9862,0.9997
19,Full_Tuning_Sampler,S-14,25,0.9789,0.9848,0.9873,0.9824,0.9991
21,Full_Tuning_Sampler,S-16,25,0.9789,0.9848,0.9873,0.9824,0.9991
18,Full_Tuning_Sampler,S-13,23,0.9672,0.9746,0.9755,0.9739,0.9986
20,Full_Tuning_Sampler,S-15,23,0.9672,0.9746,0.9755,0.9739,0.9986
26,Unfreeze_Last_Layer,S-09,22,0.8820,0.8997,0.9004,0.9018,0.9783
33,Unfreeze_Last_Layer_Sampler,S-10,20,0.8688,0.8968,0.9105,0.8852,0.9695


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import torch
import torch.nn.functional as F

# Pastikan list kelas ini sesuai dengan urutan index dataset (0, 1, 2, 3)
class_names = ["Mild Demented", "Moderate Demented", "Non Demented", "Very Mild Demented"]

def evaluate_and_visualize_all(folder_path, method_name):
    if not os.path.exists(folder_path):
        print(f"Folder tidak ditemukan: {folder_path}")
        return

    model_files = [f for f in os.listdir(folder_path) if f.endswith('.pth')]

    # Lewati jika folder kosong
    if len(model_files) == 0:
        print(f"Folder {method_name} kosong. Tidak ada model .pth yang ditemukan.")
        return

    model_files.sort() # Urutkan file model

    for filename in model_files:
        filepath = os.path.join(folder_path, filename)

        # Header pemisah antar model agar outputnya rapi
        print("\n" + "="*80)
        print(f"🚀 FOLDER/METODE: {method_name.upper()} | MODEL: {filename}")
        print("="*80)

        # 1. LOAD WEIGHTS
        model.load_state_dict(torch.load(filepath, map_location=device))
        model.eval() # Pastikan mode evaluasi

        # 2. INFERENCE
        all_labels = []
        all_preds = []
        all_probs = []

        with torch.no_grad():
            # PERBAIKAN: Unpacking disesuaikan dengan dataset ResNet50 (images, labels, idx)
            for images, labels, _ in test_loader:
                images = images.to(device)
                outputs = model(images)

                probs = F.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        all_labels = np.array(all_labels)
        all_preds = np.array(all_preds)
        all_probs = np.array(all_probs)

        # 3. CLASSIFICATION REPORT
        print("\n[1] CLASSIFICATION REPORT")
        print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))

        # 4. CONFUSION MATRIX
        print("\n[2] CONFUSION MATRIX & ROC CURVE")
        fig, axes = plt.subplots(1, 2, figsize=(16, 6)) # Jejerkan 2 grafik kiri-kanan

        # -- Grafik Kiri: Confusion Matrix --
        cm = confusion_matrix(all_labels, all_preds)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, ax=axes[0])
        axes[0].set_xlabel("Predicted", fontweight='bold')
        axes[0].set_ylabel("Actual", fontweight='bold')
        axes[0].set_title(f"Confusion Matrix: {filename[:20]}...", fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)

        # -- Grafik Kanan: ROC Curve --
        y_bin = label_binarize(all_labels, classes=[0, 1, 2, 3])
        colors = ['blue', 'red', 'green', 'orange']

        for i, color in zip(range(4), colors):
            fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
            roc_auc = auc(fpr, tpr)
            axes[1].plot(fpr, tpr, color=color, lw=2,
                         label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

        axes[1].plot([0, 1], [0, 1], 'k--', lw=2)
        axes[1].set_xlim([0.0, 1.0])
        axes[1].set_ylim([0.0, 1.05])
        axes[1].set_xlabel('False Positive Rate', fontweight='bold')
        axes[1].set_ylabel('True Positive Rate', fontweight='bold')
        axes[1].set_title('ROC Curve', fontweight='bold')
        axes[1].legend(loc="lower right")
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show() # Tampilkan gambar di Colab

        # SANGAT PENTING: Bebaskan memori Matplotlib setelah ditampilkan!
        plt.close('all')

# ==========================================
# EKSEKUSI DINAMIS UNTUK KESELURUHAN FOLDER RESNET50
# ==========================================
ROOT_DIR = '/content/drive/MyDrive/Alzheimer_MRI_Models'

print(f"\nMencari folder eksperimen di dalam: {ROOT_DIR}")
if os.path.exists(ROOT_DIR):
    # Ambil semua subfolder yang ada di dalam root directory
    subfolders = [f.path for f in os.scandir(ROOT_DIR) if f.is_dir()]
    subfolders.sort() # Urutkan nama folder sesuai abjad

    for folder_path in subfolders:
        method_name = os.path.basename(folder_path) # Ambil nama folder (Misal: Freeze_All_Sampler)

        print(f"\n\n{'*'*80}")
        print(f"MEMASUKI FOLDER: {method_name.upper()}")
        print(f"{'*'*80}")

        # Panggil fungsi evaluasi visual untuk setiap folder yang ditemukan
        evaluate_and_visualize_all(folder_path, method_name)

else:
    print(f"Root folder tidak ditemukan: {ROOT_DIR}")